# 📘 Notebook – Tabela 2

**Realização de exames preventivos segundo tipo de cobertura**
PNS 2013 e 2019
(Apenas mulheres que realizaram o exame)

## Setup do ambiente

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Setup do ambiente
sys.path.append(str(Path("..").resolve()))
from service.pns_service import (
    get_dataframe,
    list_variables,
    register_derived_variable
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## Inspeção das variáveis disponíveis (bronze)

In [2]:
df_vars = list_variables()

df_vars[df_vars["variable"].str.contains(
    "preventivo_|mamografia_", case=False, na=False
)]

,variable,descricao,categoria,tipo_dado,regra_derivacao,depends_on,code_2013,type_2013,exists_2013,code_2019,type_2019,exists_2019
10,mamografia_cobertura,Tipo de cobertura da mamografia,derivada,float,"Calculada a partir de: mamografia_sus, mamogra...","[mamografia_sus, mamografia_plano, mamografia_...",None,None,False,None,None,False
11,mamografia_fez,Indicador (0/1) se a mulher realizou exame de ...,derivada,float,Calculada a partir de: mamografia_quando,[mamografia_quando],None,None,False,None,None,False
12,mamografia_paga,A Sra pagou algum valor pela última mamografia...,fisica,string,None,None,R019,string,True,R019,string,True
13,mamografia_plano,A última mamografia foi coberta por plano de s...,fisica,string,None,None,R018,string,True,None,string,False
14,mamografia_quando,\n Quando foi a última vez que a Sra fe...,fisica,string,None,None,R017,string,True,R01701,string,True
15,mamografia_sus,A última mamografia foi feita pelo SUS? (1=Sim...,fisica,string,None,None,R020,string,True,R020,string,True
17,preventivo_cobertura,Tipo de cobertura do papanicolau,derivada,float,"Calculada a partir de: preventivo_sus, prevent...","[preventivo_sus, preventivo_plano, preventivo_...",None,None,False,None,None,False
18,preventivo_fez,Indicador (0/1) se a mulher realizou exame pre...,derivada,float,Calculada a partir de: preventivo_quando,[preventivo_quando],None,None,False,None,None,False
19,preventivo_pago,A Sra pagou algum valor pelo último exame prev...,fisica,string,None,None,R004,string,True,R004,string,True
20,preventivo_plano,O último exame preventivo foi coberto por plan...,fisica,string,None,None,R003,string,True,None,string,False


## Carregamento mínimo necessário (bronze + gold)
Objetivo: trazer **somente o que é necessário** para a Tabela 2.

In [3]:
variables = [
    # Variáveis GOLD (já persistidas)
    "preventivo_fez",
    "mamografia_fez",
    
    # Cobertura do papanicolau
    "preventivo_sus",
    "preventivo_plano",
    "preventivo_pago",
    
    # Cobertura da mamografia
    "mamografia_sus",
    "mamografia_plano",
    "mamografia_paga"
]

sources = ["2013", "2019"]

df_raw = get_dataframe(
    variables=variables,
    sources=sources
)

df_raw.head()

,preventivo_fez,id_upa,mamografia_fez,mamografia_paga,preventivo_sus,id_morador,preventivo_plano,origem,id_domicilio,mamografia_sus,mamografia_plano,preventivo_pago
0,1,1100001,0,não informado,1,1,não,2013,0001,None,não informado,não
1,0,1100001,0,não informado,None,1,não informado,2013,0002,None,não informado,não informado
2,1,1100001,0,não informado,1,2,não,2013,0003,None,não informado,não
3,1,1100001,0,não informado,1,1,não,2013,0009,None,não informado,não
4,0,1100001,0,não informado,None,2,não informado,2013,0010,None,não informado,não informado


## Conferência estrutural e de domínio (obrigatória)

In [4]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158912 entries, 0 to 158911
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   preventivo_fez    158912 non-null  object
 1   id_upa            158912 non-null  object
 2   mamografia_fez    158912 non-null  object
 3   mamografia_paga   158910 non-null  object
 4   preventivo_sus    65715 non-null   object
 5   id_morador        158912 non-null  object
 6   preventivo_plano  64351 non-null   object
 7   origem            158912 non-null  object
 8   id_domicilio      158912 non-null  object
 9   mamografia_sus    33476 non-null   object
 10  mamografia_plano  64351 non-null   object
 11  preventivo_pago   158910 non-null  object
dtypes: object(12)
memory usage: 14.5+ MB


In [5]:
cobertura_cols = [
    "preventivo_sus", "preventivo_plano", "preventivo_pago",
    "mamografia_sus", "mamografia_plano", "mamografia_paga",
    "preventivo_fez", "mamografia_fez"
]

for col in cobertura_cols:
    print(f"\nDistribuição de valores em '{col}':")
    print(df_raw[col].value_counts(dropna=False))


Distribuição de valores em 'preventivo_sus':
preventivo_sus
None    93197
1       36719
2       28560
3         436
Name: count, dtype: int64

Distribuição de valores em 'preventivo_plano':
preventivo_plano
None             94561
não informado    37757
não              18796
sim               7798
Name: count, dtype: int64

Distribuição de valores em 'preventivo_pago':
preventivo_pago
não informado    93195
não              50637
sim              15078
None                 2
Name: count, dtype: int64

Distribuição de valores em 'mamografia_sus':
mamografia_sus
None    125436
2        17363
1        16028
3           85
Name: count, dtype: int64

Distribuição de valores em 'mamografia_plano':
mamografia_plano
None             94561
não informado    51807
não               7392
sim               5152
Name: count, dtype: int64

Distribuição de valores em 'mamografia_paga':
mamografia_paga
não informado    125434
não               25477
sim                7999
None                  2
Name

## Subconjuntos analíticos 

In [6]:
df_papa = df_raw[df_raw["preventivo_fez"] == "1"].copy()
df_mamo = df_raw[df_raw["mamografia_fez"] == "1"].copy()

len(df_papa), len(df_mamo)

(65715, 33476)

## Registro das variáveis GOLD de cobertura (consolidada)
### Papanicolau

In [7]:
register_derived_variable(
    name="preventivo_cobertura",
    description=(
        "Tipo de cobertura do exame preventivo (papanicolau). "
        "Categorias: SUS, Plano, Pagou. "
        "Derivada a partir das flags preventivo_sus, preventivo_plano e preventivo_pago. "
        "Compatível com diferenças de disponibilidade entre 2013 e 2019."
    ),
    depends_on=["preventivo_sus", "preventivo_plano", "preventivo_pago"],
    func=lambda df: np.select(
        [
            df.get("preventivo_sus").eq("1"),
            df.get("preventivo_plano").eq("sim"),
            df.get("preventivo_pago").eq("sim"),
        ],
        ["SUS", "Plano", "Pagou"],
        default=None
    )
)

### Mamografia

In [8]:
register_derived_variable(
    name="mamografia_cobertura",
    description=(
        "Tipo de cobertura do exame de mamografia. "
        "Categorias: SUS, Plano, Pagou. "
        "Derivada a partir das flags mamografia_sus, mamografia_plano e mamografia_paga. "
        "Compatível com diferenças de disponibilidade entre 2013 e 2019."
    ),
    depends_on=["mamografia_sus", "mamografia_plano", "mamografia_paga"],
    func=lambda df: np.select(
        [
            df.get("mamografia_sus").eq("1"),
            df.get("mamografia_plano").eq("sim"),
            df.get("mamografia_paga").eq("sim"),
        ],
        ["SUS", "Plano", "Pagou"],
        default=None
    )
)

## Forçar cálculo e persistência no SQLite

In [9]:
df_gold = get_dataframe(
    variables=[
        "preventivo_fez",
        "mamografia_fez",
        "preventivo_cobertura",
        "mamografia_cobertura"
    ],
    sources=["2013", "2019"]
)

df_gold.head()

,preventivo_fez,preventivo_cobertura,id_upa,mamografia_fez,id_morador,origem,id_domicilio,mamografia_cobertura
0,1,SUS,1100001,0,1,2013,0001,None
1,0,None,1100001,0,1,2013,0002,None
2,1,SUS,1100001,0,2,2013,0003,None
3,1,SUS,1100001,0,1,2013,0009,None
4,0,None,1100001,0,2,2013,0010,None


## Validação empírica pós-derivação

In [10]:
print("Cobertura – Papanicolau:")
print(
    df_gold[df_gold["preventivo_fez"] == "1"]
    ["preventivo_cobertura"]
    .value_counts(dropna=False)
)

print("\nCobertura – Mamografia:")
print(
    df_gold[df_gold["mamografia_fez"] == "1"]
    ["mamografia_cobertura"]
    .value_counts(dropna=False)
)

Cobertura – Papanicolau:
preventivo_cobertura
SUS      36719
Pagou    13192
None      8640
Plano     7164
Name: count, dtype: int64

Cobertura – Mamografia:
mamografia_cobertura
SUS      16028
Pagou     6871
None      5728
Plano     4849
Name: count, dtype: int64


## Função auxiliar (percentuais)

In [11]:
def tabela_cobertura(df: pd.DataFrame, col: str) -> pd.Series:
    categorias = ['SUS', 'Plano', 'Pagou']
    dist = df[col].value_counts(normalize=True) * 100
    dist = dist.round(2)
    dist.loc["Total"] = 100.0
    return dist

## Construção da Tabela 2 (por ano)

In [12]:
tabelas = {}

for ano in sorted(df_gold["origem"].unique()):
    df_ano = df_gold[df_gold["origem"] == ano]
    
    tabela = pd.DataFrame({
        "Papanicolau": tabela_cobertura(
            df_ano[df_ano["preventivo_fez"] == "1"],
            "preventivo_cobertura"
        ),
        "Mamografia": tabela_cobertura(
            df_ano[df_ano["mamografia_fez"] == "1"],
            "mamografia_cobertura"
        )
    })
    
    tabelas[ano] = tabela

## Exibição final

In [13]:
for ano, tabela in tabelas.items():
    print(f"\nTabela 2 – Tipo de cobertura dos exames (%), PNS {ano}")
    display(tabela)


Tabela 2 – Tipo de cobertura dos exames (%), PNS 2013


,Papanicolau,Mamografia
SUS,59.16,48.8
Plano,27.87,39.4
Pagou,12.97,11.8
Total,100.00,100.0



Tabela 2 – Tipo de cobertura dos exames (%), PNS 2019


,Papanicolau,Mamografia
SUS,68.58,64.91
Pagou,31.42,35.09
Total,100.00,100.00


**Nota metodológica sobre a variável “plano de saúde” em 2019**

Na Pesquisa Nacional de Saúde (PNS) de 2019, as variáveis que identificam explicitamente se o exame preventivo (papanicolau ou mamografia) foi realizado por meio de plano de saúde não estão disponíveis no microdado, diferentemente do observado na PNS 2013. Essa ausência não decorre de falha no processamento dos dados, mas de uma mudança no instrumento de coleta e na estrutura do questionário aplicada pelo IBGE, o que inviabiliza a observação direta dessa modalidade de cobertura em 2019.

Diante disso, a estratégia adotada neste trabalho foi construir a variável de tipo de cobertura de forma compatível entre os anos, respeitando estritamente a informação observável em cada origem. Para 2019, a categoria “plano de saúde” é atribuída apenas quando explicitamente identificada no microdado; quando essa informação não está disponível, a observação permanece sem classificação nessa categoria, evitando imputações ou inferências indiretas. Essa decisão preserva a comparabilidade temporal, a transparência metodológica e a integridade empírica dos resultados, ainda que implique menor detalhamento da cobertura em 2019.